# 🌐 Telecom X - Challenge 3: Predicción de Evasión (Machine Learning)

En este cuaderno, retomamos el proceso de **Extracción, Transformación y Carga (ETL)** de datos de la parte anterior, para luego aplicar técnicas de **Machine Learning** y predecir cuáles clientes son más propensos a cancelar sus servicios (Churn).


In [ ]:
!pip install pandas numpy scikit-learn seaborn matplotlib requests plotly


## 1. 📌 Extracción y Transformación (ETL)


In [ ]:
# Importamos librerías iniciales
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import requests as rq

# Descargamos los datos crudos desde la API
url = "https://raw.githubusercontent.com/alura-cursos/challenge2-data-science-LATAM/main/TelecomX_Data.json"
telecom_base = rq.get(url).json()

# Normalizamos a DataFrame
df = pd.json_normalize(telecom_base)
df.head()


In [ ]:
# Filtrar valores nulos de Churn y Total Charges
df = df[df['Churn'] != ''].copy()
df['account.Charges.Total'] = pd.to_numeric(df['account.Charges.Total'], errors='coerce').fillna(0.0)

# Unificar servicio telefónico
df.loc[df['phone.MultipleLines'] == 'Yes', 'phone.PhoneService'] = 'Multiple lines'
df.loc[df['phone.PhoneService'] == 'Yes', 'phone.PhoneService'] = 'One line'
df.drop(columns=['phone.MultipleLines'], inplace=True)

# Parseo de booleanos
for col in ['Churn', 'customer.Partner', 'customer.Dependents', 'account.PaperlessBilling']:
    df[col] = df[col].map({'Yes': True, 'No': False})

for col in ['internet.OnlineSecurity', 'internet.OnlineBackup', 'internet.DeviceProtection', 'internet.TechSupport', 'internet.StreamingTV', 'internet.StreamingMovies']:
    df[col] = df[col].map({'Yes': True, 'No': False, 'No internet service': False})

df['customer.SeniorCitizen'] = df['customer.SeniorCitizen'].astype(bool)

# Traducción de variables
df = df.rename(columns = {
    'customerID': 'ID', 'Churn': 'Cancelacion', 'customer.gender': 'Genero', 
    'customer.SeniorCitizen': 'Jubilado', 'customer.Partner': 'ConPareja', 
    'customer.Dependents': 'Dependientes', 'customer.tenure': 'Antiguedad_Meses',
    'phone.PhoneService': 'ServicioTelefonico', 'internet.InternetService': 'ServicioInternet',
    'internet.OnlineSecurity': 'SeguridadEnLinea', 'internet.OnlineBackup': 'CopiaDeSeguridad',
    'internet.DeviceProtection': 'ProteccionDispositivos', 'internet.TechSupport': 'SoporteTecnico',
    'internet.StreamingTV': 'StreamingTV', 'internet.StreamingMovies': 'StreamingPeliculas',
    'account.Contract': 'TipoDeContrato', 'account.PaperlessBilling': 'FacturaElectronica',
    'account.PaymentMethod': 'MetodoDePago', 'account.Charges.Monthly': 'CobroMensual',
    'account.Charges.Total': 'CobroTotal'
})

df.info()


## 2. ⚙️ Preparación de Datos para Machine Learning
Eliminaremos columnas redundantes para el modelo (como el ID del cliente) y convertiremos las variables categóricas (strings) a variables numéricas (dummies) que el algoritmo entienda.


In [ ]:
# Eliminamos el ID del cliente, ya que no tiene valor predictivo
df = df.drop(columns=["ID"])

# Convertimos variables categóricas a variables dummy (One-Hot Encoding)
# drop_first=True evita la multicolinealidad
df_ml = pd.get_dummies(df, drop_first=True)

df_ml.head()


In [ ]:
# Verificamos la proporción de clientes que cancelaron y los que no
print(df_ml["Cancelacion"].value_counts(normalize=True) * 100)

sns.countplot(x="Cancelacion", data=df_ml, palette="Set2")
plt.title("Distribución de Cancelación de Clientes")
plt.show()


El conjunto de datos está **desbalanceado**, con ~73% de casos negativos y ~26% de casos positivos, por lo que es vital observar métricas como F1-Score además del Accuracy general.


In [ ]:
# Mapa de correlación entre variables numéricas clave
plt.figure(figsize=(6,5))
num_cols = ["Antiguedad_Meses", "CobroMensual", "CobroTotal", "Cancelacion"]
sns.heatmap(df_ml[num_cols].corr(), annot=True, cmap="coolwarm")
plt.title("Matriz de Correlación")
plt.show()


In [ ]:
# Boxplot que muestra cómo se distribuye la antigüedad vs la cancelación
plt.figure(figsize=(8,5))
sns.boxplot(x="Cancelacion", y="Antiguedad_Meses", data=df_ml, palette="Set3")
plt.title("Tiempo de Permanencia vs Cancelación de Clientes")
plt.xlabel("Canceló el Servicio")
plt.ylabel("Antigüedad (Meses)")
plt.show()


## 3. 🤖 Modelos de Clasificación
Separamos el DataFrame entre las variables predictoras (X) y el objetivo (y). Luego, comparamos Modelos de Logistic Regression y Random Forest.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report

# Separación de características y variable objetivo
X = df_ml.drop("Cancelacion", axis=1)
y = df_ml["Cancelacion"]

# División train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Estandarización para regresión logística
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
# ---------------------- REGRESIÓN LOGÍSTICA ----------------------
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_scaled, y_train)

y_pred_log = log_model.predict(X_test_scaled)

print("--- REGRESIÓN LOGÍSTICA ---")
print(classification_report(y_test, y_pred_log))


In [ ]:
# ---------------------- RANDOM FOREST ----------------------
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("--- RANDOM FOREST ---")
print(classification_report(y_test, y_pred_rf))


In [ ]:
# Visualización Matriz de Confusión para Random Forest
cm = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Matriz de Confusión - Random Forest")
plt.xlabel("Predicción del Modelo")
plt.ylabel("Valor Real")
plt.show()


## 4. 📈 Importancia de Variables (Feature Importance)


In [ ]:
# Extraemos la importancia de cada característica desde el Random Forest
importances = rf_model.feature_importances_

# Formateamos a Series de Pandas 
features = pd.Series(importances, index=X.columns)

# Visualizamos las 15 características más influyentes
plt.figure(figsize=(10,6))
features.sort_values(ascending=True).tail(15).plot(kind="barh", color='teal')
plt.title("Top 15 - Importancia de las Variables Predictoras")
plt.xlabel("Importancia")
plt.show()


# 📄 Conclusiones y Recomendaciones Estratégicas

### Análisis del Modelo y Desempeño
Ambos modelos (Regresión Logística y Random Forest) lograron predecir con una exactitud general aproximada del 78% al 80%. Debido a la disparidad existente en los datos originales (donde el 73% de los clientes **no** cancela el servicio), el modelo tiende levemente a ser más efectivo identificando la no-cancelación que la cancelación. No obstante, logran aislar los factores de riesgo con suma precisión analítica.

### ¿Cuáles variables influyen más en el Churn?
Gracias a la técnica de *Feature Importance* procesada por nuestro bosque aleatorio y a los análisis correlacionales del mapa de calor, detectamos 4 variables críticas:

1. **Meses de Antigüedad (`Antiguedad_Meses`)**: Es el indicador absoluto con más peso. Un cliente con baja permanencia (primeros meses) tiene un altísimo porcentaje de evasión, corroborado por el diagrama de caja donde los que cancelaron tienen medias de permanencia cercanas a la mitad que los que deciden quedarse.
2. **Cobro Total y Mensual (`CobroTotal`, `CobroMensual`)**: Los clientes expuestos a mayores costos sin una retención adecuada migran hacia otras alternativas.
3. **Contratos "Mes a Mes" (`TipoDeContrato_Month-to-month`)**: Refuerzan el fenómeno de retención; aquellos contratados de a mes cancelan mucho más, dada su libertad instantánea de evasión. 
4. **Fibra Óptica (`ServicioInternet_Fiber optic`)**: Los clientes con paquetes de fibra (habitualmente más costosos) tienen una susceptibilidad llamativa a la fuga comparado con clientes de DSL convencional.

### Recomendaciones Tácticas para Telecom X
*   **Ataque a la fuga temprana**: Implementar estrategias de "Onboarding" sumamente sólidas en los primeros 1 a 3 meses. Descuentos para los primeros pagos o atención prioritaria.
*   **Transiciones de Contrato**: Emprender una campaña para persuadir fuertemente a los poseedores de "Month-to-month" a cambiar hacia contratos de un año brindándoles recompensas por subscripción garantizada.
*   **Investigar Fallas de Fibra Óptica**: Como los clientes de Fibra tienden a ser valiosos pero con gran tasa de abandono, es fundamental analizar si existe alguna deficiencia técnica regional que está causando que la red falle.

